In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer
)

# ==========================================
# 1. ENVIRONMENT & PATH SETUP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define your paths here
IMAGE_DIR = "/content/drive/MyDrive/TrainingDataV2/Imgdir"
MASK_DIR = "/content/drive/MyDrive/TrainingDataV2/MaskDir"
OUTPUT_DIR = "/content/drive/MyDrive/segformer-paintings-v3"

# ==========================================





Using device: cuda


In [ ]:
# 2. RGB COLOR PALETTE & MAPPING (15 Classes)
# ==========================================
COLOR_MAP = {
    (0, 0, 0): 0,        # background
    (128, 0, 0): 1,      # building
    (0, 128, 0): 2,      # earth
    (128, 128, 0): 3,    # grass
    (0, 0, 128): 4,      # animal
    (128, 0, 128): 5,    # mountain
    (0, 128, 128): 6,    # path_road
    (128, 128, 128): 7,  # person
    (64, 0, 0): 8,       # rock
    (192, 0, 0): 9,      # shrub_bush
    (64, 128, 0): 10,    # sky
    (192, 128, 0): 11,   # tree_conical
    (64, 0, 128): 12,    # tree_broadleaf
    (192, 0, 128): 13,   # wooded_mass
    (64, 128, 128): 14   # Water
}

class_names = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]

id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}
num_labels = len(class_names)

def rgb_to_id(mask_pil):
    """Renkli maskeyi tıkır tıkır ID matrisine çevirir"""
    mask_np = np.array(mask_pil.convert("RGB"))
    h, w, _ = mask_np.shape
    id_mask = np.zeros((h, w), dtype=np.int64) # Varsayılan olarak 0 (background)

    for rgb, idx in COLOR_MAP.items():
        if idx == 0: continue # Zaten sıfırla doldurduk
        match = (mask_np[:, :, 0] == rgb[0]) & (mask_np[:, :, 1] == rgb[1]) & (mask_np[:, :, 2] == rgb[2])
        id_mask[match] = idx

    return Image.fromarray(id_mask.astype(np.uint8))

In [ ]:
# 3. PYTORCH DATASET WITH RGB CONVERSION
# ==========================================
class PaintingRGBDataset(Dataset):
    def __init__(self, image_dir, mask_dir, processor):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.processor = processor

        # Alfabetik sıralama (1, 10, 11, 12... şeklinde iki tarafı da eşit doğrular)
        self.img_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith('.png')])

        assert len(self.img_files) == len(self.mask_files), "Görüntü ve Maske sayıları uyuşmuyor!"

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        image = Image.open(img_path).convert("RGB")
        color_mask = Image.open(mask_path)

        # Renkli maskeyi modele vermeden önce ID maskesine dönüştür
        id_mask = rgb_to_id(color_mask)

        encoded_inputs = self.processor(images=image, segmentation_maps=id_mask, return_tensors="pt")

        for k, v in encoded_inputs.items():
            encoded_inputs[k] = v.squeeze(0)

        return encoded_inputs

In [ ]:
# 4. INITIALIZE PROCESSOR & MODEL
# ==========================================
model_checkpoint = "nvidia/segformer-b0-finetuned-ade-512-512"

processor = SegformerImageProcessor.from_pretrained(model_checkpoint)
processor.reduce_labels = False

model = SegformerForSemanticSegmentation.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
).to(device)

train_dataset = PaintingRGBDataset(IMAGE_DIR, MASK_DIR, processor)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                    
------------------------------+----------+----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([15, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([15])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
# 5. TRAINING ARGUMENTS & EXECUTION (Birebir Orijinal Ayarların)
# ==========================================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=f"{OUTPUT_DIR}/runs",
    learning_rate=6e-5,
    num_train_epochs=100,               # 100 epoch orijinal ayar
    per_device_train_batch_size=2,
    save_strategy="epoch",
    logging_steps=5,
    remove_unused_columns=False,
    fp16=True,                          # Renkli maskede fp16 güvenle açılabilir
    report_to="tensorboard"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("Starting SegFormer V3 training loop with RGB mapping pipeline...")
trainer.train()

# Model ve işlemciyi kalıcı olarak Drive'a yedekle
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"[SUCCESS] Model weights completely saved to: {OUTPUT_DIR}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting SegFormer V3 training loop with RGB mapping pipeline...


Step,Training Loss
5,2.647325
10,2.472105
15,2.348716
20,2.217939
25,2.137051
30,2.037078
35,1.967193
40,1.868311
45,1.878221
50,1.840886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[SUCCESS] Model weights completely saved to: /content/drive/MyDrive/segformer-paintings-v3


In [ ]:
import os
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Eğittiğin ve ağırlıklarını kaydettiğin güncel v3 klasörün
MODEL_DIR = "/content/drive/MyDrive/segformer-paintings-v3"
# Deneme yapacağın yeni resimlerin klasörü
TEST_IMAGE_DIR = "/content/drive/MyDrive/SemanticSegmentationV1/ProjectImages"
# Sonuçların (maskeli resimlerin) kaydedileceği yer
OUTPUT_RESULTS_DIR = "/content/drive/MyDrive/SemanticSegmentationV1/InferenceResults"

os.makedirs(OUTPUT_RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. MODEL VE İŞLEMCİYI YÜKLE
# ==========================================
print("Güncel SegFormer v3 modeli Drive'dan yükleniyor...")
processor = SegformerImageProcessor.from_pretrained(MODEL_DIR)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_DIR).to(device)
model.eval() # Modeli değerlendirme (test) moduna alıyoruz

# Sınıf listesi ve renk paleti (Senin verdiğin Pascal VOC renkleri)
class_names = model.config.id2label # Otomatik olarak modelden sınıfları çeker

# Görselleştirme için renk paleti (RGB)
# Her ID için senin CVAT'ta belirlediğin renkleri matrise basacağız
PALETTE = np.array([
    [0, 0, 0],        # background
    [128, 0, 0],      # building
    [0, 128, 0],      # earth
    [128, 128, 0],    # grass
    [0, 0, 128],      # animal
    [128, 0, 128],    # mountain
    [0, 128, 128],    # path_road
    [128, 128, 128],  # person
    [64, 0, 0],       # rock
    [192, 0, 0],      # shrub_bush
    [64, 128, 0],     # sky
    [192, 128, 0],    # tree_conical
    [64, 0, 128],     # tree_broadleaf
    [192, 0, 128],    # wooded_mass
    [64, 128, 128]    # Water
])

# ==========================================
# 3. TEST RESİMLERİNİ İŞLEME VE YÜZDE HESAPLAMA
# ==========================================
test_images = sorted([f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"\n{len(test_images)} adet resim üzerinde deneme başlatılıyor...\n")

for img_name in test_images:
    img_path = os.path.join(TEST_IMAGE_DIR, img_name)
    image = Image.open(img_path).convert("RGB")

    # Resmi modelin anlayacağı formata getir
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # Çıktıyı orijinal resim boyutuna ölçeklendir
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image.size[::-1], # (Height, Width)
        mode="bilinear",
        align_corners=False
    )

    # En yüksek olasılıklı sınıf ID'lerini al (Argmax)
    pred_seg = upsampled_logits.argmax(dim=1)[0].cpu().numpy()

    # ------------------------------------------
    # YÜZDE HESAPLAMA KISMI
    # ------------------------------------------
    total_pixels = pred_seg.size
    unique_ids, counts = np.unique(pred_seg, return_counts=True)
    pixel_counts = dict(zip(unique_ids, counts))

    print("-" * 50)
    print(f"RESİM: {img_name}")
    print("-" * 50)

    # Her bir sınıfın yüzdesini hesapla ve ekrana yazdır
    for class_id in range(len(class_names)):
        count = pixel_counts.get(class_id, 0)
        percentage = (count / total_pixels) * 100
        class_name = class_names[class_id]

        # Sadece resimde %0'dan fazla yer kaplayan sınıfları yazdıralım (Kalabalık yapmasın)
        if percentage > 0.0:
            print(f"  * {class_name:<15}: %{percentage:.2f} ({count} piksel)")

    # ------------------------------------------
    # MASKELİ GÖRSELİ OLUŞTURMA VE KAYDETME
    # ------------------------------------------
    # ID matrisini renkli RGB maskeye dönüştür
    color_seg = np.zeros((pred_seg.shape[0], pred_seg.shape[1], 3), dtype=np.uint8)
    for label, color in enumerate(PALETTE):
        color_seg[pred_seg == label] = color

    # Orijinal resim ile maskeyi yarı şeffaf overlay (harmanlama) yap
    img_np = np.array(image)
    overlay = (img_np * 0.5 + color_seg * 0.5).astype(np.uint8)

    # Yan yana görselleştirme (Sol: Orijinal, Sağ: Tahmin Maskesi)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image)
    axes[0].set_title("Orijinal Resim")
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("Model Tahmini (Overlay)")
    axes[1].axis("off")

    # Görseli Drive'a kaydet
    save_path = os.path.join(OUTPUT_RESULTS_DIR, f"result_{img_name}")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

print(f"\n[BAŞARI] Tüm resimlerin maskeli çıktıları şu klasöre kaydedildi: {OUTPUT_RESULTS_DIR}")

Güncel SegFormer v3 modeli Drive'dan yükleniyor...


/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]


20 adet resim üzerinde deneme başlatılıyor...

--------------------------------------------------
RESİM: N-0109-00-000032-wpu.jpg
--------------------------------------------------
  * background     : %24.58 (130595 piksel)
  * earth          : %0.70 (3727 piksel)
  * animal         : %5.28 (28051 piksel)
  * rock           : %0.54 (2873 piksel)
  * shrub_bush     : %7.28 (38656 piksel)
  * sky            : %11.84 (62898 piksel)
  * tree_broadleaf : %39.47 (209689 piksel)
  * Water          : %10.30 (54711 piksel)
--------------------------------------------------
RESİM: N-0134-00-000011-wpu.jpg
--------------------------------------------------
  * background     : %27.42 (130524 piksel)
  * building       : %8.78 (41784 piksel)
  * earth          : %6.70 (31869 piksel)
  * grass          : %1.19 (5669 piksel)
  * shrub_bush     : %6.98 (33228 piksel)
  * sky            : %26.02 (123864 piksel)
  * tree_broadleaf : %22.10 (105195 piksel)
  * Water          : %0.81 (3867 piksel)
----

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    Mask2FormerImageProcessor,
    Mask2FormerForUniversalSegmentation
)

# ==========================================
# 1. AYARLAR VE PATH TANIMLAMALARI
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_TYPE = "segformer"

MODEL_DIR = "/content/drive/MyDrive/segformer-paintings-v3"
TEST_IMAGE_DIR = "/content/drive/MyDrive/TestData/TestImages"
TEST_MASK_DIR = "/content/drive/MyDrive/TestData/TestMasks"

RESULTS_DIR = os.path.join(MODEL_DIR, "Test_Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==========================================
# 2. YENİ RENK HARİTASI (CVAT'IN YENİ VERDİĞİ RENKLER)
# Modelin ID sıralamasına göre ayarlandı!
# ==========================================
COLOR_MAP = {
    (0, 0, 0): 0,          # background
    (110, 13, 13): 1,      # building
    (96, 66, 7): 2,        # earth
    (131, 224, 112): 3,    # grass
    (240, 120, 240): 4,    # animal
    (37, 70, 103): 5,      # mountain
    (230, 209, 168): 6,    # path_road
    (184, 61, 245): 7,     # person
    (65, 93, 125): 8,      # rock
    (48, 173, 48): 9,      # shrub_bush
    (18, 206, 242): 10,    # sky
    (13, 135, 53): 11,     # tree_conical
    (135, 246, 171): 12,   # tree_broadleaf
    (253, 164, 5): 13,     # wooded_mass
    (85, 144, 203): 14     # Water
}

class_names = [
    "background", "building", "earth", "grass", "animal",
    "mountain", "path_road", "person", "rock", "shrub_bush",
    "sky", "tree_conical", "tree_broadleaf", "wooded_mass", "Water"
]
num_classes = len(class_names)

def rgb_to_id(mask_pil):
    mask_np = np.array(mask_pil.convert("RGB"))
    h, w, _ = mask_np.shape
    id_mask = np.zeros((h, w), dtype=np.int64)
    for rgb, idx in COLOR_MAP.items():
        if idx == 0: continue
        match = (mask_np[:, :, 0] == rgb[0]) & (mask_np[:, :, 1] == rgb[1]) & (mask_np[:, :, 2] == rgb[2])
        id_mask[match] = idx
    return id_mask

# ==========================================
# 3. MODELİ YÜKLE
# ==========================================
print(f"\n[{MODEL_TYPE.upper()}] Modeli test için yükleniyor...")
if MODEL_TYPE == "segformer":
    processor = SegformerImageProcessor.from_pretrained(MODEL_DIR)
    model = SegformerForSemanticSegmentation.from_pretrained(MODEL_DIR).to(device)
else:
    processor = Mask2FormerImageProcessor.from_pretrained(MODEL_DIR)
    model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_DIR).to(device)
model.eval()

# ==========================================
# 4. ÇIKARIM (INFERENCE) VE KARIŞIKLIK MATRİSİ
# ==========================================
img_files = sorted([f for f in os.listdir(TEST_IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
mask_files = sorted([f for f in os.listdir(TEST_MASK_DIR) if f.lower().endswith('.png')])

assert len(img_files) == len(mask_files), "HATA: Resim ve Maske sayıları eşit değil!"

confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

print(f"{len(img_files)} adet resim üzerinde piksel piksel test yapılıyor. Lütfen bekleyin...")

for img_name, mask_name in zip(img_files, mask_files):
    image = Image.open(os.path.join(TEST_IMAGE_DIR, img_name)).convert("RGB")
    gt_id_mask = rgb_to_id(Image.open(os.path.join(TEST_MASK_DIR, mask_name)))

    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    if MODEL_TYPE == "segformer":
        upsampled_logits = torch.nn.functional.interpolate(
            outputs.logits, size=image.size[::-1], mode="bilinear", align_corners=False
        )
        pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy()
    else:
        predicted_semantic_maps = processor.post_process_semantic_segmentation(
            outputs, target_sizes=[image.size[::-1]]
        )
        pred_mask = predicted_semantic_maps[0].cpu().numpy()

    flat_gt = gt_id_mask.flatten()
    flat_pred = pred_mask.flatten()
    indices = (flat_gt >= 0) & (flat_gt < num_classes)

    confusion_matrix += np.bincount(
        num_classes * flat_gt[indices] + flat_pred[indices],
        minlength=num_classes**2
    ).reshape(num_classes, num_classes)

# ==========================================
# 5. METRİKLERİ HESAPLA VE KAYDET
# ==========================================
print("\n" + "="*60 + f"\n{MODEL_TYPE.upper()} - METRİK HESAPLAMALARI\n" + "="*60)

tp = np.diag(confusion_matrix)
fp = np.sum(confusion_matrix, axis=0) - tp
fn = np.sum(confusion_matrix, axis=1) - tp
eps = 1e-15

iou_per_class = tp / (tp + fp + fn + eps)
precision_per_class = tp / (tp + fp + eps)
recall_per_class = tp / (tp + fn + eps)

total_pixels_per_class = np.sum(confusion_matrix, axis=1)
total_valid_pixels = np.sum(total_pixels_per_class)
class_frequencies = total_pixels_per_class / (total_valid_pixels + eps)
fwiou = np.sum(class_frequencies * iou_per_class)

# Verileri CSV için listeye ekleme ve Ekrana Yazdırma
results_data = []
present_classes_iou = []

for i, name in enumerate(class_names):
    if total_pixels_per_class[i] > 0:
        iou_val = iou_per_class[i] * 100
        prec_val = precision_per_class[i] * 100
        rec_val = recall_per_class[i] * 100
        present_classes_iou.append(iou_per_class[i])

        print(f"Sınıf: {name:<15} | IoU: %{iou_val:05.2f} | P: %{prec_val:05.2f} | R: %{rec_val:05.2f}")

        results_data.append({
            "Class Name": name,
            "IoU (%)": round(iou_val, 2),
            "Precision (%)": round(prec_val, 2),
            "Recall (%)": round(rec_val, 2)
        })

miou = np.mean(present_classes_iou) * 100
mean_precision = np.mean([precision_per_class[i] for i in range(num_classes) if total_pixels_per_class[i] > 0]) * 100
mean_recall = np.mean([recall_per_class[i] for i in range(num_classes) if total_pixels_per_class[i] > 0]) * 100
fwiou_percent = fwiou * 100

print("-" * 60)
print(f"GENEL mIoU (Mean IoU)                : %{miou:.2f}")
print(f"GENEL FWIoU (Ağırlıklı IoU)          : %{fwiou_percent:.2f}")
print(f"GENEL MEAN PRECISION (Hassasiyet)    : %{mean_precision:.2f}")
print(f"GENEL MEAN RECALL (Duyarlılık)       : %{mean_recall:.2f}")
print("=" * 60)

# CSV DOSYASINA KAYDET
df = pd.DataFrame(results_data)
csv_path = os.path.join(RESULTS_DIR, f"{MODEL_TYPE}_class_metrics.csv")
df.to_csv(csv_path, index=False)

# TXT RAPOR DOSYASINA KAYDET
txt_path = os.path.join(RESULTS_DIR, f"{MODEL_TYPE}_overall_report.txt")
with open(txt_path, "w") as f:
    f.write(f"--- {MODEL_TYPE.upper()} 6-IMAGE TEST REPORT ---\n")
    f.write(f"Mean IoU (mIoU)       : {miou:.2f}%\n")
    f.write(f"Freq Weighted IoU     : {fwiou_percent:.2f}%\n")
    f.write(f"Mean Precision        : {mean_precision:.2f}%\n")
    f.write(f"Mean Recall           : {mean_recall:.2f}%\n")

print(f"\n[BAŞARILI] İşlem tamamlandı! Sonuçlar Drive'a kaydedildi.")


[SEGFORMER] Modeli test için yükleniyor...


/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'reduce_labels'
  image_processor = cls(**image_processor_dict)


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

6 adet resim üzerinde piksel piksel test yapılıyor. Lütfen bekleyin...

SEGFORMER - METRİK HESAPLAMALARI
Sınıf: background      | IoU: %31.37 | P: %39.21 | R: %61.07
Sınıf: building        | IoU: %48.34 | P: %87.42 | R: %51.95
Sınıf: earth           | IoU: %20.10 | P: %57.21 | R: %23.66
Sınıf: grass           | IoU: %34.48 | P: %55.10 | R: %47.95
Sınıf: animal          | IoU: %50.06 | P: %56.54 | R: %81.37
Sınıf: mountain        | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: path_road       | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: person          | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: rock            | IoU: %51.68 | P: %72.86 | R: %64.00
Sınıf: shrub_bush      | IoU: %09.76 | P: %18.41 | R: %17.21
Sınıf: sky             | IoU: %94.50 | P: %95.36 | R: %99.05
Sınıf: tree_conical    | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: tree_broadleaf  | IoU: %30.38 | P: %31.75 | R: %87.59
Sınıf: wooded_mass     | IoU: %00.00 | P: %00.00 | R: %00.00
Sınıf: Water           | IoU: %43.32 | P: